# KratTrade: Hybrid Energy Portfolio Optimisation
## Toy example
A four-hour example with a battery, a solar asset and two markets, followed by a six-variable QUBO and a QAOA experiment.


## S01 — Selected problem and proposed toy example
### Selected problem (from the company application)
> Batteries, solar and wind parks and demand response are managed as one portfolio. For every hour, we need to decide which asset to offer to which market, taking into account weather forecast uncertainty and imbalance costs. The number of possible combinations grows quickly as more assets are added.

### Proposed toy example
Select one four-hour battery schedule and one solar-backed energy sales commitment. Evaluate their combined expected profit under two solar production scenarios.

Compare the exact best plan pair with QAOA results in simulation and on IBM Quantum and IQM.


## S02 — Terms
| Term | Meaning here |
|---|---|
| MW | Power: the rate of charging, discharging or reserved response. |
| MWh | Energy; 1 MW sustained for one hour is 1 MWh. |
| Reserve capacity | Power held available for a possible request from the grid operator. |
| Imbalance shortfall | Committed energy that the portfolio does not deliver. |
| Scenario | One possible solar production trajectory over the same four hours. |
| QUBO | A quadratic cost function of binary variables. |
| QAOA | A parameterised quantum circuit whose angles are adjusted by a classical optimiser. |


## S03 — Toy-model assumptions
- One battery, one solar asset and four one-hour periods.
- Two revenue sources: energy sales and upward reserve capacity.
- Two solar scenarios, each with probability 0.5.
- All sales commitments, charging, discharging and reserve schedules are fixed before solar production is known. The same schedules apply in both scenarios.
- Grid charging is allowed at the tabulated energy price. Charging purchases are scheduled and settled separately from the sales commitment.
- Solar production and scheduled battery discharge jointly meet the energy sales commitment. Any surplus is settled at 50% of the tabulated energy price; any shortfall costs 150 EUR/MWh. Advance commitments therefore offer a better price than surplus settlement but expose the portfolio to shortfall costs.
- Reserve bids are accepted at the tabulated capacity price. Reserved power cannot also be used for scheduled discharge.
- This is a **capacity-only reserve example**: no reserve activation occurs in the evaluated scenarios. Each reserve commitment must nevertheless be deliverable for a 15-minute upward response.
- Use a conservative battery deliverability test: enough energy must be available at both the start and end of each hour to support its reserved power for 15 minutes. Each hour is checked independently, not as a sequence of reserve activations.
- No degradation charges, network fees or shared grid export limit. Simultaneous battery charging and discharging is excluded. Battery energy must return to 1 MWh at the end of the horizon.

These assumptions define the calculations below; no additional market rules are needed for this example.


## S04 — Synthetic input data
All figures are illustrative, not market observations. The energy price is used for advance sales and scheduled charging purchases. Surplus energy is settled at half that price; this discount is a toy assumption, not a description of Baltic market settlement.

| Hour | Energy price (EUR/MWh) | Reserve price (EUR/MW/hour) | Solar: expected (MWh) | Solar: low (MWh) | Shortfall cost (EUR/MWh) |
|---|---:|---:|---:|---:|---:|
| 12–13 | 40 | 10 | 2 | 1 | 150 |
| 13–14 | 30 | 15 | 3 | 1.5 | 150 |
| 14–15 | 70 | 25 | 2 | 1 | 150 |
| 15–16 | 90 | 40 | 1 | 0.5 | 150 |

| Battery parameter | Value |
|---|---:|
| Energy capacity | 2 MWh |
| Initial and terminal energy | 1 MWh |
| Maximum charge / discharge power | 1 MW / 1 MW |
| Charge / discharge efficiency | 0.95 / 0.95 |
| Reserve response duration | 0.25 hours |


## S05 — Numeric candidate plans and profit calculation
Each vector contains one value for each of the four hours.

### Battery candidates
| Candidate | Charge (MW) | Discharge (MW) | Upward reserve (MW) |
|---|---|---|---|
| B1 | [0.5, 0, 0, 0] | [0, 0, 0, 0.45125] | [0, 0, 0, 0] |
| B2 | [0, 0, 0, 0] | [0, 0, 0, 0] | [0.5, 0.5, 0.5, 0.5] |
| B3 | [0.25, 0, 0, 0] | [0, 0, 0.225625, 0] | [0.25, 0.25, 0.25, 0.25] |

The discharge amounts account for both charging and discharging losses and return the battery to its initial energy.

### Solar-backed sales commitments
| Candidate | Committed energy (MWh) |
|---|---|
| G1 | [2, 3, 2, 1] |
| G2 | [1, 1.5, 1, 0.5] |
| G3 | [1, 1.5, 1, 1] |

These are portfolio energy commitments: battery discharge can cover a solar shortfall. Its remaining discharge is sold as surplus; do not also count it as a separate sale.

### Worked example: B2 + G2
B2 holds 0.5 MW of reserve in every hour, with no charging or discharging. G2 commits [1, 1.5, 1, 0.5] MWh to energy sales.

In the first hour, the committed sale earns 1 MWh x 40 EUR/MWh = 40 EUR, and the reserve earns 0.5 MW x 10 EUR/MW/hour x 1 hour = 5 EUR.
- Expected solar scenario: production is 2 MWh. The extra 1 MWh earns 20 EUR at the surplus price, giving 65 EUR in total.
- Low solar scenario: production is 1 MWh. It exactly meets the commitment, giving 45 EUR in total.
- With equal scenario probabilities, expected profit for this hour is (65 + 45) / 2 = 55 EUR.

Repeating the same calculation for all four hours:

| Hour | Expected solar scenario profit (EUR) | Low solar scenario profit (EUR) | Probability-weighted profit (EUR) |
|---|---:|---:|---:|
| 12-13 | 65 | 45 | 55 |
| 13-14 | 75 | 52.5 | 63.75 |
| 14-15 | 117.5 | 82.5 | 100 |
| 15-16 | 87.5 | 65 | 76.25 |
| **Four-hour total** | **345** | **245** | **295** |

There are no shortfalls, charging purchases or reserve activations for this pair. The 295 EUR is its expected four-hour profit, not a third scenario. Comparing all nine pairs establishes that B2 + G2 is the best pair in this candidate library (S07).

The following equations express the feasibility checks and profit calculation for any candidate pair.

### Feasibility
For hour $t$, charge $c_t$, discharge $d_t$, reserve $r_t$, and end-of-hour battery energy $e_t$:
$$
e_t=e_{t-1}+0.95c_t\Delta t-\frac{d_t\Delta t}{0.95},\qquad \Delta t=1\text{ hour}.
$$
Check $0\le e_t\le2$, $0\le c_t,d_t\le1$, $c_td_t=0$, $d_t+r_t\le1$, and $e_4=1$.
For the conservative reserve test:
$$
\min(e_{t-1},e_t)\ge\frac{0.25r_t}{0.95},\qquad r_t\ge0.
$$
The listed schedules satisfy these checks. Their charging hours have zero scheduled discharge; the conservative reserve rule does not credit stopping charge as additional reserve capability.

### Scenario profit
For selected battery plan $i$, sales plan $j$, scenario $s$, solar energy $a_{t,s}$, and commitment $q_{j,t}$, define delivered energy before shortfall settlement:
$$
y_{i,t,s}=a_{t,s}+d_{i,t}\Delta t.
$$
With advance-sale and charging price $p_t$, surplus settlement price $u_t=0.5p_t$, reserve price $v_t$, and shortfall rate $k_t=150$:
$$
\Pi_{ij,s}=\sum_t\left[
p_tq_{j,t}
+u_t\max(y_{i,t,s}-q_{j,t},0)
-k_t\max(q_{j,t}-y_{i,t,s},0)
-p_tc_{i,t}\Delta t
+v_tr_{i,t}\Delta t
\right].
$$
Compute expected profit as $\bar\Pi_{ij}=0.5\Pi_{ij,\mathrm{expected}}+0.5\Pi_{ij,\mathrm{low}}$.
The maximum terms are evaluated classically for each plan pair before constructing the QUBO.


## S06 — Six-variable QUBO
Assign one binary variable to each candidate:
$$
b_1,b_2,b_3,g_1,g_2,g_3\in\{0,1\}.
$$
Exactly one battery plan and one sales plan must be selected.
With $C_{ij}=-\bar\Pi_{ij}$, minimise
$$
E(b,g)=\sum_{i,j}C_{ij}b_ig_j+
\lambda\left(\sum_i b_i-1\right)^2+
\lambda\left(\sum_j g_j-1\right)^2.
$$
Expanding the squares produces only constant, linear and quadratic terms because $x^2=x$ for binary variables.

For this example, choose $\lambda=1+\sum_{i,j}|C_{ij}|$, with the constant 1 expressed in the same numerical cost units. Every invalid bitstring incurs at least one penalty. The bound makes invalid states more expensive than the valid optimum. Smaller penalties may be tested after checking the full energy landscape.

Verify all 64 bitstrings against the nine valid plan pairs. Keep the constant offset when comparing QUBO energy to economic cost.

**Circuit size:** Six logical qubits, no slack variables. Nine bitstrings satisfy the two one-hot constraints. The quantum solver searches the fixed candidate library, not all possible continuous schedules.


## S07 — Exact classical reference
1. Evaluate the two-scenario profit of each of the nine plan pairs.
2. Select the pair with the highest expected profit.
3. Enumerate all 64 QUBO bitstrings and confirm that its minimum corresponds to an optimal valid pair.
4. Check decoding, constraint violations, signs and constant offsets.

Report the optimal expected profit and all tied optimal pairs. This is the exact reference **within the specified candidate library**.

The classical angle optimiser used by QAOA is a separate component: it adjusts circuit parameters rather than providing this reference solution.

### Verified reference values (EUR)
| Battery plan | G1 | G2 | G3 |
|---|---:|---:|---:|
| B1 | 123.996875 | 250.306250 | 270.246875 |
| B2 | 145.000000 | 295.000000 | 291.250000 |
| B3 | 133.370313 | 270.396875 | 266.646875 |

The exact best pair is B2 + G2, with expected profit 295.000000 EUR. All 64 QUBO bitstrings were checked against this reference. These values are classical arithmetic checks, not QAOA results.


## S08 — From QUBO to a QAOA circuit
Once the binary variables and QUBO coefficients are defined, Qiskit can construct the cost operator and parameterised QAOA circuit using standard tools. You do not need to write the individual quantum gates by hand to build this example.

### Model first, circuit second
QUBO is a mathematical model, not a quantum algorithm. It expresses a cost using binary variables, linear terms and pairwise products. QAOA is one method for searching for low-cost assignments.

In this example, we compute all nine plan-pair profits before constructing the QUBO. That table already makes the classical optimum easy to find. This is an introductory circuit and modelling exercise.

For a larger problem, define variables, economic costs and operational constraints directly where possible, rather than enumerating every complete solution. Derive and validate the QUBO from those rules; choosing a cost function is not simply trying arbitrary formulas until one produces attractive results. Discretisation, penalty weights and additional variables may be needed.

### What the circuit does
- Prepare an equal superposition of the 64 six-bit assignments. Only nine obey our selection rules.
- Apply a **cost layer**: each assignment receives a phase determined by its cost and an angle $\gamma$. This does not output a numerical cost or, on its own, change measurement probabilities.
- Apply a **mixer layer**, controlled by $\beta$. Together with the cost phases, it creates interference that changes the probabilities of measuring assignments.
- Repeat this pair of layers $p$ times, then measure. A measured bitstring specifies which plans were selected; its cost is evaluated classically.

For standard QAOA, $p=1$ has two adjustable angles, one $\gamma$ and one $\beta$, shared across the corresponding layers. $p=2$ has four angles. We choose $p$; the classical optimiser adjusts the angles. Layer count is different from both compiled gate depth and the number of measurement repetitions.

The cost function is therefore used both to construct the circuit and to score its measured outputs. The aim is to obtain good valid assignments more frequently; finding the exact optimum is not guaranteed.


### Classical optimisation with COBYLA
**COBYLA is a classical, derivative-free optimiser.** It uses evaluated objective values to propose new circuit angles. It runs outside the quantum circuit; it is not an intermediate quantum measurement or another quantum layer.

A typical loop is:
1. Choose an initial set of angles.
2. Execute the circuit many times at those same angles, for example 1,000 shots.
3. Calculate the QUBO cost of each measured bitstring, including penalties, and average the costs.
4. Return the average to COBYLA, which proposes another set of angles.
5. Repeat until the chosen stopping criterion or evaluation budget is reached.

One submission can request many shots. New angle evaluations generally require further execution requests. A provider session can reduce scheduling overhead where available, but the optimiser remains classical.

Start with $p=1$ and compare with $p=2$, using several initial angle settings. More layers can increase expressive power but also gate count, noise and optimisation effort.

### Minimal Qiskit circuit example
The next cell encodes the expanded QUBO and builds a six-qubit, one-layer QAOA circuit. The profit table comes from S07. Run the short setup cell first in Google Colab. It installs Qiskit and Qiskit Optimization only if the required imports fail. The following cell builds the circuit.

**This cell only constructs the circuit.** It does not run COBYLA, sample a simulator, submit a hardware job or solve the optimisation problem. Those steps remain in the S09 implementation cells.

For IBM–IQM tests, keep the logical objective identical and compile separately for each device. Compare fixed simulator-trained angles first; report any hardware-in-the-loop training separately. Record shots, all optimiser evaluations, initialisations, compiled depth, two-qubit gate count and total execution cost.


In [1]:
# Run this cell before the Qiskit example in Google Colab.
try:
    from qiskit_optimization import QuadraticProgram
    from qiskit.circuit.library import QAOAAnsatz
except ImportError:
    %pip install -q "qiskit>=2,<3" "qiskit-optimization>=0.7,<0.8"
    from qiskit_optimization import QuadraticProgram
    from qiskit.circuit.library import QAOAAnsatz

print("Qiskit is ready.")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Qiskit is ready.


In [2]:
import numpy as np
from itertools import combinations
from qiskit_optimization import QuadraticProgram
from qiskit.circuit.library import QAOAAnsatz

# Rows: B1, B2, B3. Columns: G1, G2, G3.
profit = np.array([
    [123.996875, 250.306250, 270.246875],
    [145.000000, 295.000000, 291.250000],
    [133.3703125, 270.396875, 266.646875],
])
C = -profit
penalty = 1.0 + np.abs(C).sum()

qubo = QuadraticProgram("KratTrade")
B = ["b1", "b2", "b3"]
G = ["g1", "g2", "g3"]
for name in B + G:
    qubo.binary_var(name)

# For binary variables, expanding lambda * (sum(x) - 1)**2
# gives constant lambda, linear terms -lambda,
# and pairwise terms 2*lambda within each group.
linear = {name: -penalty for name in B + G}
quadratic = {
    (B[i], G[j]): float(C[i, j])
    for i in range(3)
    for j in range(3)
}
for group in [B, G]:
    for x, y in combinations(group, 2):
        quadratic[(x, y)] = 2 * penalty

qubo.minimize(
    constant=2 * penalty,
    linear=linear,
    quadratic=quadratic,
)

# Verify that the expanded objective matches S06 for all 64 states.
from itertools import product

for bits in product([0, 1], repeat=6):
    expected_cost = (
        sum(C[i, j] * bits[i] * bits[j + 3]
            for i in range(3) for j in range(3))
        + penalty * (sum(bits[:3]) - 1)**2
        + penalty * (sum(bits[3:]) - 1)**2
    )
    assert np.isclose(qubo.objective.evaluate(list(bits)), expected_cost)

cost_operator, offset = qubo.to_ising()
circuit = QAOAAnsatz(
    cost_operator=cost_operator,
    reps=1,
    flatten=True,
)

assert circuit.num_qubits == 6
assert circuit.num_parameters == 2
print("Qubits:", circuit.num_qubits)
print("Adjustable angles:", circuit.num_parameters)
print(circuit.decompose().draw(output="text"))

# offset is a constant energy shift. Keep it when comparing absolute
# objective values; it does not change which bitstring is optimal.
# This is an unmeasured circuit template, not an execution or optimiser loop.


Qubits: 6
Adjustable angles: 2
global phase: 5118.535156250001*γ[0]
     ┌────────────┐┌───────────────────────────────┐     »
q_0: ┤ U(π/2,0,π) ├┤ P((-1724.9390625000003)*γ[0]) ├──■──»
     ├────────────┤├───────────────────────────────┤┌─┴─┐»
q_1: ┤ U(π/2,0,π) ├┤ P((-1681.5890625000002)*γ[0]) ├┤ X ├»
     ├────────────┤├───────────────────────────────┤└───┘»
q_2: ┤ U(π/2,0,π) ├┤ P((-1712.0070312500002)*γ[0]) ├─────»
     ├────────────┤├───────────────────────────────┤     »
q_3: ┤ U(π/2,0,π) ├┤ P((-1846.0304687500002)*γ[0]) ├─────»
     ├────────────┤├───────────────────────────────┤     »
q_4: ┤ U(π/2,0,π) ├┤ P((-1639.3625000000002)*γ[0]) ├─────»
     ├────────────┤└──┬─────────────────────────┬──┘     »
q_5: ┤ U(π/2,0,π) ├───┤ P((-1633.1421875)*γ[0]) ├────────»
     └────────────┘   └─────────────────────────┘        »
«                                                                             »
«q_0: ─────────────────────────────────■────■─────────────────────────────────»
«    

## S09 — Implementation
The following cells organise the calculations and experiments.


In [ ]:
# TODO: Define the S04 data and battery parameters.


In [ ]:
# TODO: Encode S05 candidates, validate battery schedules, and compute the 3 x 3 expected-profit table.


In [ ]:
# TODO: Build S06 QUBO and validate all 64 bitstrings against exact plan-pair enumeration.


In [ ]:
# TODO: Run simulator QAOA with COBYLA for p=1 and p=2; decode samples.


In [ ]:
# TODO: Execute the same logical experiment on IBM and IQM; record compiled resources.


In [ ]:
# TODO: Compare feasibility, expected profit, optimum frequency, time and cost.


## S10 — Evaluation
| Method | Qubits | p | Valid sample rate | Best valid expected profit | Gap to exact optimum | Optimal-pair frequency | Execution time | Total cost |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Exact enumeration | — | — | — | | 0 | — | | |
| QAOA simulator | 6 | 1 | | | | | | |
| IBM | 6 | 1 | | | | | | |
| IQM | 6 | 1 | | | | | | |

Repeat quantum rows for p = 2 and additional initialisations.

- Report profit differences in EUR, independently of penalty energy.
- Measure the valid sample rate before discarding invalid outcomes.
- Report the fraction of all samples that encode an optimal valid pair.
- Separate compilation, parameter search, device execution and queue time.
- Count all shots and optimiser calls, not just the final measurement.

This example tests model correctness and quantum solution quality. Nine plan pairs are easy to enumerate classically; the experiment does not establish a runtime or economic advantage.

For commercial payback, the relevant future metric would be additional realised profit over a strong classical method minus additional computing and operating costs.


## S11 — References
- [PennyLane: Intro to QAOA](https://pennylane.ai/demos/tutorial_qaoa_intro)
- [IBM: QAOA tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/quantum-approximate-optimization-algorithm)
- [Qiskit Optimization: Minimum Eigen Optimizer](https://qiskit-community.github.io/qiskit-optimization/tutorials/03_minimum_eigen_optimizer.html)
- [IQM: Qiskit user guide](https://docs.iqm.tech/iqm-client/user_guide_qiskit.html)
- [IQM QAOA](https://docs.iqm.tech/iqm-qaoa/)
- [QAOA theory and implementation tutorial](https://arxiv.org/abs/2301.09535)
